# Processing Protected Areas (WDPA)
In this notebook, we will process the World Database on Protected Areas (WDPA) to calculate national protected areas. The WDPA is a comprehensive global database of terrestrial and marine protected areas, updated on a monthly basis. The database is compiled from a wide range of sources, including governments, non-governmental organizations, and individual researchers. The WDPA is managed by the United Nations Environment Programme World Conservation Monitoring Centre (UNEP-WCMC) and the International Union for Conservation of Nature (IUCN).

## [Methodology](https://www.protectedplanet.net/en/resources/calculating-protected-area-coverage)

1. [Start downloading](01_data_download.ipynb) the latest WDPA monthly release.
2. The WDPA can be filtered to exclude records.
3. A buffer is created around protected areas reported as points using their Reported Area. **There are important caveats associated with this method, some of which are explored by Visconti et al. 2013. Buffering points can underestimate or overestimate protected area coverage as the circles created around points might cover areas where protected areas do not exist (overestimation) or overlap with areas where other protected areas already exist (underestimation). It can also give inaccurate values for sites that are partly terrestrial and marine as the absence of boundaries make it difficult to predict which portion of a protected area is in the land or the sea.**
4. Both polygon and buffered point layers are combined in a single layer.
5. The layer above is flattened (dissolved) by country/territory – to remove overlaps between designations within countries/territories and avoid double counting (please note that this retains overlaps between countries and should therefore only be used to calculate national, not regional or global, coverage.


## Setup

### Library import

In [ ]:
import logging
import os
import subprocess
from pathlib import Path

import geopandas as gpd
import pandas as pd

In [ ]:
# Create a logger
logger = logging.getLogger(__name__)

# Set the log level to INFO
logger.setLevel(logging.INFO)

### Utils

In [ ]:
def calculate_radius(rep_area):
    """
    Calculate the radius of a circle based on the reported area.

    Parameters:
    rep_area (float): The reported area in square meters.
    """
    return (rep_area / 3.14159265358979323846) ** 0.5

**create_mbtiles**

In [ ]:
def create_mbtiles(
    source_path: Path,
    output_path: Path,
    layer_name: str,
    max_zoom: int,
    opts="--read-parallel --no-tile-compression -s EPSG:4326 -B4",
):
    """
    Use tippecanoe to create pbf tiles at dest_path from source_path (geojson).
    layer_name is used for the name of the layer in the MBTILE.
    Regex file path (/*.geojson) is supported for source_path.
    This function replaces the previous two functions (create_mbtiles & mbtile_to_pbf).

    More info: https://github.com/mapbox/tippecanoe#options

    Args:
        source_path (Path): path to source geojson
        output_path (Path): path to output .mbtiles
        layer_name (str): name of layer in the MBTILE
        max_zoom (int): max zoom level
        opts (str): options for tippecanoe

    Returns:
        (int): 0 if the file was created successfully, 1 if the file creation failed.
    """
    try:
        opts += f" -z{max_zoom}"
        cmd = f"tippecanoe -o {output_path} -l {layer_name} {opts} {source_path}"
        logger.info(f"Processing: {cmd}")
        r = subprocess.call(cmd, shell=True)
        if r == 0:
            logger.info("Task created")
        return r

    except Exception as e:
        logger.error(e)
        return 1

## Processing the WDPA data
### 1. Read data:

In [ ]:
path_in = "../data/raw/"
path_out = "../data/processed/"

In [ ]:
point0 = gpd.read_file(
    os.path.join(
        path_in
        + "WDPA_Apr2024_Public_shp/WDPA_Apr2024_Public_shp_0/WDPA_Apr2024_Public_shp-points.shp"
    ),
    engine="pyogrio",
    use_arrow=True,
    where="MARINE='0' AND STATUS!='Not Reported' AND \
        STATUS!='Proposed' AND DESIG_ENG NOT LIKE '%MAB%' AND REP_AREA!=0",
)
poly0 = gpd.read_file(
    os.path.join(
        path_in
        + "WDPA_Apr2024_Public_shp/WDPA_Apr2024_Public_shp_0/WDPA_Apr2024_Public_shp-polygons.shp"
    ),
    engine="pyogrio",
    use_arrow=True,
    where="MARINE='0' AND STATUS!='Not Reported' AND \
        STATUS!='Proposed' AND DESIG_ENG NOT LIKE '%MAB%'",
)
point1 = gpd.read_file(
    os.path.join(
        path_in
        + "WDPA_Apr2024_Public_shp/WDPA_Apr2024_Public_shp_1/WDPA_Apr2024_Public_shp-points.shp"
    ),
    engine="pyogrio",
    use_arrow=True,
    where="MARINE='0' AND STATUS!='Not Reported' AND \
        STATUS!='Proposed' AND DESIG_ENG NOT LIKE '%MAB%' AND REP_AREA!=0",
)
poly1 = gpd.read_file(
    os.path.join(
        path_in
        + "WDPA_Apr2024_Public_shp/WDPA_Apr2024_Public_shp_1/WDPA_Apr2024_Public_shp-polygons.shp"
    ),
    engine="pyogrio",
    use_arrow=True,
    where="MARINE='0' AND STATUS!='Not Reported' AND \
        STATUS!='Proposed' AND DESIG_ENG NOT LIKE '%MAB%'",
)
point2 = gpd.read_file(
    os.path.join(
        path_in
        + "WDPA_Apr2024_Public_shp/WDPA_Apr2024_Public_shp_2/WDPA_Apr2024_Public_shp-points.shp"
    ),
    engine="pyogrio",
    use_arrow=True,
    where="MARINE='0' AND STATUS!='Not Reported' AND \
        STATUS!='Proposed' AND DESIG_ENG NOT LIKE '%MAB%' AND REP_AREA!=0",
)
poly2 = gpd.read_file(
    os.path.join(
        path_in
        + "WDPA_Apr2024_Public_shp/WDPA_Apr2024_Public_shp_2/WDPA_Apr2024_Public_shp-polygons.shp"
    ),
    engine="pyogrio",
    use_arrow=True,
    where="MARINE='0' AND STATUS!='Not Reported' AND \
        STATUS!='Proposed' AND DESIG_ENG NOT LIKE '%MAB%'",
)

### 2. Filter WDPA to exclude records:
- MARINE: We keep only 0 (predominantly or entirely terrestrial) records.
- "Non Reported" protected areas and "Proposed"
- MAB (Note: MAB sites reported as OECMs are included in coverage analyses)
- Sites submitted as points with no reported area

In [ ]:
dataframes = [poly0, point0, poly1, point1, poly2, point2]

# for i, df in enumerate(dataframes):
#    # Take rows where 'MARINE' is equal to '0'
#    df = df[df["MARINE"] == "0"]
#
#    # Remove rows where 'status' is equal to 'Not Reported'
#    df = df[(df["STATUS"] != "Not Reported") & (df["STATUS"] != "Proposed")]
#
#    # Remove rows where 'DESIG' contains 'MAB'
#    df = df[~df["DESIG_ENG"].str.contains("MAB", case=False)]
#
#    # Check if the dataframe is one of point1, point2, or point3
#    if i in [1, 3, 5]:
#        # Remove rows where reported area is 0
#        df = df[(df["REP_AREA"] != 0)]
#
#    # Update the original dataframes in the list
#    dataframes[i] = df

### 3. Create buffers around points based on reported area

In [ ]:
# Calculate radius based on REP_AREA
# Iterate through the list and process the desired dataframes
for idx in [1, 3, 5]:
    # Get the dataframe at the specified index
    gdf = dataframes[idx]

    # Reproject in Mollweide
    gdf = gdf.to_crs("ESRI:54009")

    # Transform the reported area from square kilometers to square meters
    gdf["REP_AREA_m"] = gdf["REP_AREA"] * 1000000

    # Create the "radius" column by applying the calculate_radius function to the "REP_AREA" column
    gdf["radius"] = gdf["REP_AREA_m"].apply(calculate_radius)

    # Create buffers around the points using the "radius" column
    gdf_buffered = gdf.copy()
    gdf_buffered["geometry"] = gdf.apply(lambda row: row.geometry.buffer(row["radius"]), axis=1)

    # Reproject back to WGS84
    gdf_buffered = gdf_buffered.to_crs("EPSG:4326")

    # Remove rows with invalid geometries
    gdf_buffered = gdf_buffered[gdf_buffered["geometry"].is_valid]

    # Update the original dataframe with the buffered data
    dataframes[idx] = gdf_buffered

### 4. Remove unnecessary attributes
**NOTE:** The REP_AREA field is the area of the protected area reported by the data providers. GIS_AREA refers to the WDPA area assinged by UNEP-WCMC based on the areas of the geometries. Often, the REP_AREA is not filled by the providers, so the area provided is 0, although the geometry says otherwise. For that reason, for the polygons' dataframes, we use the column GIS_AREA to get the AREA of the geometries. For the buffered points, we use the REP_AREA field as it is the only one available and the one that was used for creating the polygons.

In [ ]:
# Columns for poly datasets
dataframes[0].columns

In [ ]:
# Columns of point datasets
dataframes[1].columns

In [ ]:
# Keep only columns of interest in the dataframes, the area must be "GIS_AREA" for the polygons and
# "REP_AREA" for the points
for idx in [0, 2, 4]:
    gdf = dataframes[idx]
    gdf = gdf[
        [
            "WDPAID",
            "WDPA_PID",
            "NAME",
            "DESIG_ENG",
            "DESIG_TYPE",
            "IUCN_CAT",
            "GIS_AREA",
            "STATUS",
            "STATUS_YR",
            "ISO3",
            "geometry",
        ]
    ].rename(columns={"GIS_AREA": "AREA"})
    dataframes[idx] = gdf

for idx in [1, 3, 5]:
    gdf = dataframes[idx]
    gdf = gdf[
        [
            "WDPAID",
            "WDPA_PID",
            "NAME",
            "DESIG_ENG",
            "DESIG_TYPE",
            "IUCN_CAT",
            "REP_AREA",
            "STATUS",
            "STATUS_YR",
            "ISO3",
            "geometry",
        ]
    ].rename(columns={"REP_AREA": "AREA"})
    dataframes[idx] = gdf

In [ ]:
dataframes[0].columns

In [ ]:
dataframes[1].columns

### 4. Simplify geometries

In [ ]:
for idx in [0, 2, 4]:
    # Get the dataframe at the specified index
    gdf = dataframes[idx]

    # Simplify the geometries
    gdf["geometry"] = gdf["geometry"].simplify(tolerance=0.001, preserve_topology=True)

    dataframes[idx] = gdf

### 5. Merge the 6 datasets (polygons and buffered points) in a single layer and segregate those that are "Proposed"

In [ ]:
# Check that all of them have the same crs
first_crs = dataframes[0].crs
same_crs = all(gdf.crs == first_crs for gdf in dataframes[1:])
if same_crs:
    print("All gdf have the same crs:", first_crs)
else:
    print("gdf have different crs")

In [ ]:
# Merge dataframes
wdpa = pd.concat(dataframes)
len(wdpa)

### 6. Fix invalid geometries and save shapefile for visualization

In [ ]:
# Fix invalid geometries with buffer(0)
wdpa["geometry"] = wdpa["geometry"].buffer(0)

In [ ]:
# Check if there are invalid geometries
invalid_geoms = wdpa[~wdpa["geometry"].is_valid]

In [ ]:
len(invalid_geoms)

In [ ]:
wdpa.to_file(os.path.join(path_out + "wdpa/wdpa.shp"), driver="ESRI Shapefile")

In [ ]:
# Open wdpa file
wdpa = gpd.read_file(os.path.join(path_out + "wdpa/wdpa.shp"))

### 7. Intersect with rangelands

In [ ]:
# Open rangelands file
rangelands = gpd.read_file(os.path.join(path_out + "ecoregions_2017.geojson"))

In [ ]:
# Select only wdpa tat intersect with rangelands
wdpa_rangelands = gpd.overlay(wdpa, rangelands, how="intersection")
len(wdpa_rangelands)

In [ ]:
# Keep only relevant column from rangelands dataset
wdpa_rangelands = wdpa_rangelands[
    [
        "WDPAID",
        "WDPA_PID",
        "NAME",
        "DESIG_ENG",
        "DESIG_TYPE",
        "IUCN_CAT",
        "AREA",
        "STATUS",
        "STATUS_YR",
        "ISO3",
        "BIOME_NAME",
        "geometry",
    ]
]

In [ ]:
# Save wdpa_rangelands
wdpa_rangelands.to_file(
    os.path.join(path_out + "wdpa/wdpa_rangelands.shp"), driver="ESRI Shapefile"
)

### 8. Dissolve WDPA polygons to calculate coverage statistics

In [ ]:
# Stats for country
!mapshaper-xl 16gb \
    {path_out}wdpa/wdpa_rangelands.shp -dissolve2 fields=ISO3 -explode -proj \
        +proj=moll +lon_0=0 +x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs +type=crs -each \
            AREA=this.area -proj EPSG:4326 -o {path_out}wdpa/wdpa_rangelands_dissolved_iso.shp

In [ ]:
# Stats for rangeland type
!mapshaper-xl 16gb \
    {path_out}wdpa/wdpa_rangelands.shp -dissolve2 fields=BIOME_NAME -explode -proj \
        +proj=moll +lon_0=0 +x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs +type=crs -each \
            AREA=this.area -proj EPSG:4326 -o {path_out}wdpa/wdpa_rangelands_dissolved_bio.shp

In [ ]:
iso = gpd.read_file(os.path.join(path_out + "wdpa/wdpa_rangelands_dissolved_iso.shp"))
bio = gpd.read_file(os.path.join(path_out + "wdpa/wdpa_rangelands_dissolved_bio.shp"))

In [ ]:
# Calculate area in km per iso3
iso["area_km"] = iso["AREA"] / 1000000
iso_area = iso.groupby("ISO3")["area_km"].sum().reset_index()
iso_area

In [ ]:
# Calculate area in km per biome
bio["area_km"] = bio["AREA"] / 1000000
bio_area = bio.groupby("BIOME_NAME")["area_km"].sum().reset_index()
bio_area

### 9. Save data in other formats

In [ ]:
wdpa.to_parquet(os.path.join(path_out + "wdpa.parquet"), index=False)

In [ ]:
wdpa.to_file(os.path.join(path_out + "wdpa.geojson"), driver="GeoJSON")

### 10. Create `MBTiles`

In [ ]:
create_mbtiles(
    os.path.join(path_out, "wdpa.geojson"),
    os.path.join(path_out, "wdpa.mbtiles"),
    "Protected Areas",
    12,
    "--force --read-parallel -zg -Z2 --drop-densest-as-needed --extend-zooms-if-still-dropping",
)